In [1]:
##### Isolated agriculutre's portion of ILO labor data 
# method described in data porcessing memo

import os
import pandas as pd
import numpy as np

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
sectors = pd.read_csv(f"{cd}/Data/Raw/FAO_ILO_labor/FAO_ILO_subsector_labor_02232026.csv")
aggregate = pd.read_csv(f"{cd}/Data/Clean/Labor/ILO_ag_labor_estimate.csv")

country_codes = pd.read_csv(
    f"{cd}/Data/Correspondence_tables/country_names.csv",
    encoding="cp1252"
)

# Set save path
save_path = f"{cd}/Data/Clean/Labor/ILO_ag_labor_estimate_adjusted.csv"


In [3]:
### Clean sub-sector labor data and get ag share of labor

# keep only needed columns 
col = ['Area Code (M49)', 'Indicator Code', 'Year', 'Value']
sectors = sectors[col]

# split into ag, foresty, and fishing 
# indicator codes: 21097 = fishing, 21111 = ag, 21093 = forestry 

fishing = sectors[sectors['Indicator Code'] == 21097]
forestry = sectors[sectors['Indicator Code'] == 21093]
agriculture = sectors[sectors['Indicator Code'] == 21111]

# fill 2020 in using latest available year and then isolate for 2020 
def fill_2020(sector, code): 

    # add 2020
    area_codes = sector['Area Code (M49)'].unique()
    year_2020_df = pd.DataFrame({
        'Area Code (M49)': area_codes,
        'Year': 2020
    })

    sector = pd.merge(
        year_2020_df,
        sector,
        on=['Area Code (M49)', 'Year'],
        how='outer'
    )

    sector = sector.sort_values(['Area Code (M49)', 'Year'])

    # Forward fill and backward fill
    sector['Value_ffill'] = sector.groupby('Area Code (M49)')['Value'].ffill()
    sector['Value_bfill'] = sector.groupby('Area Code (M49)')['Value'].bfill()

    # Keep the years where the filled values actually came from
    sector['Year_ffill'] = sector.groupby('Area Code (M49)').apply(
        lambda g: g['Year'].where(g['Value'].notna()).ffill()
    ).reset_index(level=0, drop=True)

    sector['Year_bfill'] = sector.groupby('Area Code (M49)').apply(
        lambda g: g['Year'].where(g['Value'].notna()).bfill()
    ).reset_index(level=0, drop=True)

    # Choose the closest
    target_year = 2020
    sector[f'labor_thousands_{code}'] = sector.apply(
        lambda row: row['Value_ffill'] 
            if pd.notna(row['Value_ffill']) and 
            (pd.isna(row['Value_bfill']) or abs(row['Year_ffill'] - target_year) <= abs(row['Year_bfill'] - target_year))
            else row['Value_bfill'],
        axis=1
    )

    sector = sector[sector['Year'] == 2020]

    col = ['Area Code (M49)', f'labor_thousands_{code}']
    sector = sector[col]

    return sector

fishing = fill_2020(fishing, 21097)
forestry = fill_2020(forestry, 21093)
agriculture = fill_2020(agriculture, 21111)

# remerge all sectors 
sector = agriculture.merge(forestry, on='Area Code (M49)', how='outer')
sector = sector.merge(fishing, on='Area Code (M49)', how='outer')

# drop rows missing any data 
sector = sector.dropna()

# calculate ag share of labor
sector['ag_share_labor'] = sector['labor_thousands_21111'] / (sector['labor_thousands_21111'] + sector['labor_thousands_21093'] + sector['labor_thousands_21097'])

/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_82750/2496377798.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sector['Year_ffill'] = sector.groupby('Area Code (M49)').apply(
/var/folders/48/ky2jtbmj31bfj15cr5gq480w0000gn/T/ipykernel_82750/2496377798.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sector['Year_bfill'] = sector.groupby('Area Code (M49)').apply(
/var/folders/48/ky2jtbmj

In [4]:
### Get country-level assumptions 
# for country's missing 2020 data
# using sub-regional (i.e. North Africa) average where available and regional (i.e. Africa) where not

# Filter country list to those in FAO data
country_codes = country_codes.merge(aggregate['ISO3'], on='ISO3', how='right')

# Merge with ag share data
ag_shares = sector.merge(country_codes, right_on='FAO_code', left_on='Area Code (M49)', how='right')

# get sub-region data
sub_region_count = (
    ag_shares
    .groupby('UN_subregion')['ag_share_labor']
    .count()
    .reset_index()
    .rename(columns={'ag_share_labor': 'sub_region_non_missing_count'})
)

sub_region_mean = (
    ag_shares
    .groupby('UN_subregion')['ag_share_labor']
    .mean()
    .reset_index()
    .rename(columns={'ag_share_labor': 'sub_region_mean'})
)

ag_shares = ag_shares.merge(sub_region_count, on='UN_subregion', how='left')
ag_shares = ag_shares.merge(sub_region_mean, on='UN_subregion', how='left')

# fill with sub-regional average 
# except for Micronesia - fill missing with sub-regional average for Polynesia 

def fill_value(row): 
    # If it already has a value, keep it
    if pd.notna(row['ag_share_labor']):
        return row['ag_share_labor']

    else: 
        return row['sub_region_mean']

ag_shares['ag_share_labor_filled'] = ag_shares.apply(fill_value, axis=1)

# replace Micronesia with Polynesia 

POL_mean = sub_region_mean.loc[
    sub_region_mean['UN_subregion'] == 'Polynesia', 
    'sub_region_mean'
].values[0]


ag_shares['ag_share_labor_final'] = np.where(
    ag_shares['UN_subregion'].isin(['Micronesia']), 
    POL_mean,  
    ag_shares['ag_share_labor_filled']  
)

# keep only needed vars 
col_to_keep = ['ISO3', 'ag_share_labor_final']
ag_shares = ag_shares[col_to_keep]


In [5]:
### Break out labor for ag only

# merge with labor data
aggregate = aggregate.merge(ag_shares, on='ISO3', how='outer')

# keep only 2020 data
col_to_keep = ['ISO3', 'Country_Name', '2020', 'ag_share_labor_final']
capital_stock = aggregate[col_to_keep]

aggregate = aggregate.rename(columns={
    '2020': 'AFF_labor_thousands_2020', 
    'ag_share_labor_final': 'ag_share_AFF_labor_2020'}
    )

# calculate ag labor
aggregate['ag_labor_thousands_2020'] = aggregate['AFF_labor_thousands_2020'] * aggregate['ag_share_AFF_labor_2020']

col_to_keep = ['ISO3', 'ag_labor_thousands_2020']
aggregate = aggregate[col_to_keep]

In [6]:
# Save
aggregate.to_csv(save_path, index=False)